In [1]:
from pathlib import Path
import math
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

In [4]:
# This section mirrors the ECoG loading logic from:
# Final_Code/Data_Preprocessing/ecog_data_split.ipynb
#
# Output naming convention used for the rest of this notebook:
# x_ecog_all, y_ecog_all,
# x_ecog_train, y_ecog_train,
# x_ecog_val, y_ecog_val,
# x_ecog_test, y_ecog_test


ECOG_DATA_ROOT = Path("../../Coding_and_Data")  # Change this if the dataset directory changes.
ECOG_BASE_FOLDER = "Ecog"  # Change this if the ECoG parent folder name changes.
RANDOM_STATE = 42


def _label_counts(y: np.ndarray) -> dict:
    labels, counts = np.unique(y, return_counts=True)
    return dict(zip(labels.tolist(), counts.tolist()))


def load_all_ecog_blocks(root_path: Path, base_folder: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load all aligned ECoG blocks and return:
      x_ecog_all: (n_samples, 3, 1600)
      y_ecog_all: (n_samples,)

    Assumptions for this notebook:
      - labels are already binary: 0 = no seizure, 1 = seizure
      - each CSV row is one sample
      - each sample contains 1600 time points per channel

    If the input changes from CSV files to saved tensors, this function is the main section to replace.
    """
    base_path = root_path / base_folder
    if not base_path.exists():
        raise FileNotFoundError(f"ECoG base folder not found: {base_path}")

    all_x, all_y = [], []

    for ecog_folder in sorted(base_path.iterdir()):
        if not ecog_folder.is_dir() or "Ecog" not in ecog_folder.name:
            continue

        channel_block_sets = []
        for ch in range(1, 4):
            ch_dir = ecog_folder / f"channel{ch}"  # Change this if channel folders use a different naming convention.
            if not ch_dir.exists():
                raise FileNotFoundError(f"Missing channel directory: {ch_dir}")
            blocks = {path.name for path in ch_dir.iterdir() if path.is_file() and path.name.startswith("block") and path.suffix == ".csv"}  # Change this discovery rule if file names or file type change.
            channel_block_sets.append(blocks)

        common_blocks = sorted(channel_block_sets[0] & channel_block_sets[1] & channel_block_sets[2])

        for block_name in common_blocks:
            channel_data = []
            channel_labels = []

            for ch in range(1, 4):
                block_path = ecog_folder / f"channel{ch}" / block_name  # Change this path logic if the on-disk structure changes.
                df = pd.read_csv(block_path, header=None)  # Replace this read step if inputs become .pt/.npy tensors instead of CSV.

                signals = df.iloc[:, :-1].to_numpy(dtype=np.float32)  # Replace this slice if tensor files already store signal arrays directly.
                labels = df.iloc[:, -1].to_numpy(dtype=np.int64)  # Replace this slice if tensor files already store labels separately.

                if signals.shape[1] != 1600:  # Change this check if the sequence length is not 1600 in the new data format.
                    raise ValueError(f"Expected 1600 features in {block_path}, found {signals.shape[1]}")

                channel_data.append(signals)
                channel_labels.append(labels)

            labels_stack = np.stack(channel_labels, axis=1)
            y_block = np.max(labels_stack, axis=1).astype(np.int64)
            y_block[y_block == 2] = 1 #mark 2 and 1 as 1

            if not np.isin(y_block, [0, 1]).all():
                raise ValueError(
                    f"Found non-binary labels in {ecog_folder.name}/{block_name}: {np.unique(y_block)}"
                )

            x_block = np.stack(channel_data, axis=1).astype(np.float32)

            all_x.append(x_block)
            all_y.append(y_block)

    if not all_x:
        raise RuntimeError(f"No aligned ECoG blocks found under {base_path}")

    x_ecog_all = np.vstack(all_x)
    y_ecog_all = np.concatenate(all_y)
    return x_ecog_all, y_ecog_all


x_ecog_all, y_ecog_all = load_all_ecog_blocks(ECOG_DATA_ROOT, ECOG_BASE_FOLDER)

x_ecog_train, x_ecog_temp, y_ecog_train, y_ecog_temp = train_test_split(
    x_ecog_all,
    y_ecog_all,
    test_size=0.25,
    stratify=y_ecog_all,
    random_state=RANDOM_STATE,
)

x_ecog_val, x_ecog_test, y_ecog_val, y_ecog_test = train_test_split(
    x_ecog_temp,
    y_ecog_temp,
    test_size=0.4,
    stratify=y_ecog_temp,
    random_state=RANDOM_STATE,
)

print("Loaded ECoG tensors:")
print(" - x_ecog_all:", x_ecog_all.shape)
print(" - y_ecog_all:", y_ecog_all.shape, _label_counts(y_ecog_all))
print("Split tensors:")
print(" - x_ecog_train:", x_ecog_train.shape)
print(" - y_ecog_train:", y_ecog_train.shape, _label_counts(y_ecog_train))
print(" - x_ecog_val:", x_ecog_val.shape)
print(" - y_ecog_val:", y_ecog_val.shape, _label_counts(y_ecog_val))
print(" - x_ecog_test:", x_ecog_test.shape)
print(" - y_ecog_test:", y_ecog_test.shape, _label_counts(y_ecog_test))


Loaded ECoG tensors:
 - x_ecog_all: (9344, 3, 1600)
 - y_ecog_all: (9344,) {0: 5271, 1: 4073}
Split tensors:
 - x_ecog_train: (7008, 3, 1600)
 - y_ecog_train: (7008,) {0: 3953, 1: 3055}
 - x_ecog_val: (1401, 3, 1600)
 - y_ecog_val: (1401,) {0: 790, 1: 611}
 - x_ecog_test: (935, 3, 1600)
 - y_ecog_test: (935,) {0: 528, 1: 407}


In [5]:
#Save tensors
torch.save({
    "X_train": x_ecog_train,
    "y_train": y_ecog_train,
    "X_valid": x_ecog_val,
    "y_valid": y_ecog_val,
    "X_test":  x_ecog_test,
    "y_test":  y_ecog_test
}, "ecog_tensors.pt")